In [ ]:
# Install dependencies
!pip install -q sentence-transformers pandas numpy scikit-learn matplotlib seaborn

In [ ]:
# Upload data files
from google.colab import files
import os

print("Please upload synthetic_data_for_contrastive_learning.jsonl")
uploaded = files.upload()

print("\nPlease upload dev_track_a.jsonl")
uploaded = files.upload()

print("\n✓ Files uploaded successfully!")

In [ ]:
# Import libraries
import json
from pathlib import Path
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from sentence_transformers.util import cos_sim
from datetime import datetime

print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Data loading functions
def load_contrastive_data(jsonl_path):
    """Load synthetic data for contrastive learning."""
    triplets = []
    
    print(f"Loading training data from: {jsonl_path}")
    
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            
            anchor = record.get('anchor_story')
            similar = record.get('similar_story')
            dissimilar = record.get('dissimilar_story')
            
            if anchor and similar and dissimilar:
                triplets.append((anchor, similar, dissimilar))
    
    print(f"✓ Loaded {len(triplets)} training triplets")
    return triplets

def prepare_training_examples(triplets):
    """Convert triplets to InputExample format."""
    examples = []
    
    for anchor, positive, negative in triplets:
        examples.append(InputExample(texts=[anchor, positive], label=1.0))
        examples.append(InputExample(texts=[anchor, negative], label=0.0))
    
    print(f"✓ Created {len(examples)} training examples")
    return examples

In [ ]:
# Fine-tuning function
def finetune_model(
    model_name="all-MiniLM-L6-v2",
    train_examples=None,
    batch_size=16,
    epochs=4,
    warmup_steps=100
):
    """Fine-tune a SentenceTransformer model using contrastive loss."""
    # Disable WandB logging to avoid API key issues
    import os
    os.environ["WANDB_DISABLED"] = "true"
    
    print(f"\nLoading base model: {model_name}")
    model = SentenceTransformer(model_name)
    
    # Create DataLoader
    train_dataloader = DataLoader(
        train_examples, 
        shuffle=True, 
        batch_size=batch_size
    )
    
    # Use CosineSimilarityLoss
    train_loss = losses.CosineSimilarityLoss(model)
    
    print(f"\n{'='*60}")
    print(f"Starting Fine-Tuning")
    print(f"{'='*60}")
    print(f"  Base model:         {model_name}")
    print(f"  Batch size:         {batch_size}")
    print(f"  Epochs:             {epochs}")
    print(f"  Training examples:  {len(train_examples)}")
    print(f"{'='*60}\n")
    
    # Fine-tune
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=epochs,
        warmup_steps=warmup_steps,
        output_path="finetuned_model",
        show_progress_bar=True
    )
    
    print(f"\n✓ Model saved to: finetuned_model/")
    return model

In [ ]:
# Evaluation function
def evaluate_model(model, dev_data_path):
    """Evaluate model on Track A dev set."""
    print(f"\nEvaluating on: {dev_data_path}")
    
    # Load dev data
    dev_data = []
    with open(dev_data_path, 'r', encoding='utf-8') as f:
        for line in f:
            dev_data.append(json.loads(line))
    
    print(f"✓ Loaded {len(dev_data)} dev examples")
    
    # Collect unique texts
    all_texts = []
    for record in dev_data:
        all_texts.extend([record['anchor_text'], record['text_a'], record['text_b']])
    
    unique_texts = list(set(all_texts))
    print(f"✓ Encoding {len(unique_texts)} unique texts...")
    
    # Generate embeddings
    embeddings = model.encode(unique_texts, show_progress_bar=True, convert_to_tensor=True)
    embedding_lookup = dict(zip(unique_texts, embeddings))
    
    # Evaluate
    correct = 0
    total = len(dev_data)
    
    print(f"✓ Computing predictions...")
    
    for record in dev_data:
        anchor_emb = embedding_lookup[record['anchor_text']]
        text_a_emb = embedding_lookup[record['text_a']]
        text_b_emb = embedding_lookup[record['text_b']]
        
        sim_a = cos_sim(anchor_emb, text_a_emb).item()
        sim_b = cos_sim(anchor_emb, text_b_emb).item()
        
        prediction = sim_a > sim_b
        
        if prediction == record['text_a_is_closer']:
            correct += 1
    
    accuracy = correct / total
    
    print(f"\n{'='*60}")
    print(f"Evaluation Results")
    print(f"{'='*60}")
    print(f"  Accuracy: {accuracy:.4f} ({correct}/{total})")
    print(f"{'='*60}\n")
    
    return accuracy

In [ ]:
# Main training pipeline
print("\n" + "="*60)
print("Fine-Tuned Embedding Baseline")
print("Author: Usman Amjad")
print("="*60 + "\n")

start_time = datetime.now()

# Step 1: Load training data
print("[1/4] Loading Training Data")
print("-" * 60)
triplets = load_contrastive_data("synthetic_data_for_contrastive_learning.jsonl")
train_examples = prepare_training_examples(triplets)

# Step 2: Fine-tune model
print("\n[2/4] Fine-Tuning Model")
print("-" * 60)
model = finetune_model(
    model_name="all-MiniLM-L6-v2",
    train_examples=train_examples,
    batch_size=16,
    epochs=4,
    warmup_steps=100
)

# Step 3: Evaluate on dev set
print("\n[3/4] Evaluating on Dev Set")
print("-" * 60)
accuracy = evaluate_model(model, "dev_track_a.jsonl")

# Step 4: Save results
print("\n[4/4] Saving Results")
print("-" * 60)

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

results = {
    "model": "all-MiniLM-L6-v2",
    "training_triplets": len(triplets),
    "training_samples": len(train_examples),
    "dev_accuracy": accuracy,
    "epochs": 4,
    "batch_size": 16,
    "training_duration_seconds": duration,
    "timestamp": start_time.isoformat()
}

with open("results.json", 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print(f"✓ Results saved to: results.json")
print(f"✓ Model saved to: finetuned_model/")
print(f"✓ Training duration: {duration:.2f} seconds ({duration/60:.2f} minutes)")

print("\n" + "="*60)
print("DONE! Fine-tuning and evaluation completed successfully.")
print("="*60 + "\n")

In [ ]:
# Compare with baseline
print("\n" + "="*60)
print("Comparing with Baseline Model")
print("="*60 + "\n")

# Evaluate baseline
print("Evaluating baseline (pre-trained) model...")
baseline_model = SentenceTransformer("all-MiniLM-L6-v2")
baseline_acc = evaluate_model(baseline_model, "dev_track_a.jsonl")

# Calculate improvement
absolute_improvement = accuracy - baseline_acc
relative_improvement = (absolute_improvement / baseline_acc) * 100

print("\n" + "="*60)
print("COMPARISON RESULTS")
print("="*60)
print(f"Baseline Model:       {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")
print(f"Fine-Tuned Model:     {accuracy:.4f} ({accuracy*100:.2f}%)")
print("-" * 60)
print(f"Absolute Improvement: {absolute_improvement:+.4f} ({absolute_improvement*100:+.2f} pp)")
print(f"Relative Improvement: {relative_improvement:+.2f}%")
print("="*60 + "\n")

In [ ]:
# Download results
from google.colab import files

print("Downloading results...")
files.download("results.json")

# Zip and download model
!zip -r finetuned_model.zip finetuned_model/
files.download("finetuned_model.zip")

print("\n✓ All files downloaded!")